# PGAC Phase 4 — Full-stack SAE training (11 new layers)

**Target**: train 11 TopK SAEs at layers `[15, 19, 23, 27, 35, 39, 43, 47, 51, 59, 63]`
(every-4th from L11→L63 excluding already-trained L11/L31/L55).

Combined with `caiovicentino1/qwen36-27b-sae-papergrade` (L11/L31/L55), gives 14-layer
coverage — enough to benchmark PGAC kernel end-to-end across early/mid/late routing.

**Hyperparams** (parity with papergrade):
- TopK SAE: k=128, d_sae=65536 (13× expansion), AuxK k_aux=2560, alpha=1/32
- Tokens: 200M per SAE (papergrade parity)
- Corpus: fineweb-edu 70% / OpenThoughts 20% / OpenMathInstruct 10%
- LR: cosine 2e-4 → 6e-5, 5K warmup, Adam, grad clip 1.0

**HARD RULE** (memory: `feedback_colab_must_checkpoint_or_dont_run.md`):
- Drive mount + resume — every cell that produces state must save to Drive
- Checkpoint every 10M tokens to Drive AND HF
- Resume-safe: re-run cell 11 (training loop) and it picks up where it left off

**Compute**: ~36-40h on RTX 6000 Blackwell 96GB. Cost ~$30 (R$135).
**Drive**: `/content/drive/MyDrive/openinterp_runs/pgac_phase4/`
**HF target**: `caiovicentino1/qwen36-27b-sae-fullstack`


## 1. Drive mount + resume paths (HARD RULE)


In [ ]:
from pathlib import Path
import os, json, time, gc, math, random, shutil
import torch, numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / 'pgac_phase4'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'sae_ckpt').mkdir(parents=True, exist_ok=True)
(OUT / 'logs').mkdir(parents=True, exist_ok=True)
print(f'OUT: {OUT}')
print(f'Existing: {sorted(p.name for p in OUT.iterdir())}')

LOCAL_CKPT_DIR = '/content/sae_ckpt'  # fast scratch, mirrored to Drive
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)


## 2. Install — transformers from main (qwen3_5 support)


In [ ]:
import sys, subprocess
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    has_qwen35 = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception:
    has_qwen35 = False

if not has_qwen35:
    print('Installing transformers from source for qwen3_5 support...')
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'tqdm', 'sentencepiece', 'tokenizers', 'protobuf', 'hf_transfer')
    pip('uninstall', '-y', '-q', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','-q','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]
    needs_restart = True
else:
    print(f'transformers {transformers.__version__} has qwen3_5 ✓')
    needs_restart = False

# CRITICAL: flash-linear-attention required for Qwen3.6 GDN layers fast path.
# Without it, GDN falls to torch impl = ~10× slower forward pass.
# (memory: feedback_qwen35_training_stack.md)
try:
    import fla  # noqa
    print('flash-linear-attention installed ✓')
except ImportError:
    print('Installing flash-linear-attention (required for Qwen3.6 GDN fast path)...')
    pip('install', '-q', '--no-cache-dir', 'flash-linear-attention')
    needs_restart = True

# Note: bitsandbytes NOT used. Tested bnb.AdamW8bit at d_sae=65536 → OOM at first opt.step()
# because bnb allocates state buffers as fp32 first, then quantizes — peak transient = fp32 Adam.
# torch.optim.Adam with bf16 params allocates state as bf16 directly (no transient), fits cleanly.

if needs_restart:
    print('\n*** RESTART RUNTIME NOW (Runtime → Restart session), then re-run cells 1+2. ***')

# HF auth
try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t: os.environ['HF_TOKEN'] = t
except Exception:
    t = os.environ.get('HF_TOKEN')
if t:
    from huggingface_hub import login
    login(token=t, add_to_git_credential=False)
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('HF auth OK')
else:
    print('⚠ Set HF_TOKEN in Colab secrets before checkpoint cell')


## 3. CFG — all knobs in one place


In [ ]:
# ---- Model ----
MODEL_ID      = 'Qwen/Qwen3.6-27B'
D_MODEL       = 5120

# ---- Layers ----
# 11 NEW layers (every-4th from L11→L63 excluding already-trained L11/L31/L55)
NEW_LAYERS    = [15, 19, 23, 27, 35, 39, 43, 47, 51, 59, 63]
EXISTING      = [11, 31, 55]                       # already in caiovicentino1/qwen36-27b-sae-papergrade
LAYERS        = NEW_LAYERS                         # this notebook trains the 11 NEW

# ---- SAE architecture ----
# d_sae=40960 (8x expansion) — empirically determined VRAM ceiling for 11 SAEs.
# Tested: 65536 (13x) and 49152 (9.6x) both OOM at peak ~94-95 GB on RTX 6000 96GB
# (forward intermediates + Adam state init + fragmentation eat the budget).
# 8x expansion is canonical Gemma Scope baseline; ve ~0.78-0.83 expected.
# Papergrade L11/L31/L55 used 65536 (13x) — that ran with only 3 SAEs.
N_FEATURES    = 40_960                             # 8x expansion
K_TOPK        = 128                                # Gao et al. sweet spot
K_AUX         = 2_560                              # d_model/2
ALPHA_AUX     = 1.0 / 32
DEAD_TOKENS   = 10_000_000                         # dead feature threshold

# ---- Training ----
TOKEN_BUDGET  = 200_000_000                        # per SAE (papergrade parity)
BATCH_SIZE    = 4096                               # SAE training batch (tokens)
LR_PEAK       = 2e-4
LR_FLOOR      = 6e-5
WARMUP_STEPS  = 5_000
GRAD_CLIP     = 1.0

# ---- Activation extraction ----
SEQ_LEN       = 1024                               # tokens per sequence
FWD_BATCH     = 2                                  # sequences per forward pass

# ---- Corpus mix ----
CORPUS_MIX = [
    ('HuggingFaceFW/fineweb-edu',             'sample-10BT', 0.70),
    ('open-thoughts/OpenThoughts-114k',       'default',     0.20),
    ('nvidia/OpenMathInstruct-2',             'default',     0.10),
]

# ---- Checkpointing ----
CKPT_EVERY_TOK = 10_000_000                        # every 10M tokens → Drive + HF
HF_REPO        = 'caiovicentino1/qwen36-27b-sae-fullstack'

# ---- Misc ----
DEVICE        = 'cuda'
DTYPE_MODEL   = torch.bfloat16                     # 27B model
DTYPE_SAE     = torch.bfloat16                     # bf16 SAE — required to fit 11 SAEs in 96GB
SEED          = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# Memory budget (RTX 6000 Blackwell 96GB) — d_sae=40960 + bf16 Adam:
#   Qwen3.6-27B bf16              = 54.7 GB
#   11 SAE params bf16 (d=40960)  =  9.2 GB
#   11 Adam state bf16 (matches)  = 18.4 GB
#   Forward pass intermediates    =  3-5 GB
#   1 transient grad bf16         =  0.8 GB
#   Frag overhead                 =  1-2 GB
#   Total peak                    ≈ 90 GB → fits with 6 GB margin
# Note: torch.optim.Adam is preferred over bnb.AdamW8bit because bnb allocates
# state buffers as fp32 first then quantizes — that init transient OOMs at d=65536.
# torch.zeros_like(p) preserves param dtype, so bf16 params → bf16 Adam state.

print(f'Config: n={N_FEATURES}, k={K_TOPK}, tokens/SAE={TOKEN_BUDGET/1e6:.0f}M')
print(f'New layers ({len(LAYERS)}): {LAYERS}')
print(f'(Existing in papergrade: {EXISTING})')


## 4. Load Qwen3.6-27B (bf16, sdpa)


In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE_MODEL,
    device_map=DEVICE,
    attn_implementation='sdpa',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f'VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB / 96 GB')


## 5. Layer path resolution + hook smoke test


In [ ]:
def get_layer_module(m, idx):
    for path in [('model','language_model','layers'),
                 ('language_model','layers'),
                 ('model','layers')]:
        try:
            cur = m
            for p in path:
                cur = getattr(cur, p)
            return cur[idx]
        except AttributeError:
            continue
    raise RuntimeError('Could not locate decoder layers')

for L in LAYERS:
    mod = get_layer_module(model, L)
    print(f'L{L}: {type(mod).__name__}')

# Smoke test — confirm hooks fire on all 11 layers
_probe = {}
def _mk_probe(L):
    def _h(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        _probe[L] = h.detach().float().norm().item()
    return _h
handles = [get_layer_module(model, L).register_forward_hook(_mk_probe(L)) for L in LAYERS]
ids = tok('The quick brown fox jumps over', return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
    model(ids)
for h in handles: h.remove()
print(f'Hook norms: {dict(sorted(_probe.items()))}')
assert all(v > 0 for v in _probe.values()), 'Some hooks did not fire!'
print('✓ All 11 hooks fire')


## 6. Corpus stream — fineweb-edu 70% / OpenThoughts 20% / OpenMath 10%


In [ ]:
from datasets import load_dataset

def _load_stream(repo, split):
    try:
        return load_dataset(repo, split, split='train', streaming=True)
    except Exception:
        return load_dataset(repo, split='train', streaming=True)

def _text_fineweb(row):
    return row.get('text', '')

def _text_openthoughts(row):
    conv = row.get('conversations') or row.get('messages') or []
    if not conv:
        return row.get('text', '')
    msgs = [{'role': m.get('from', m.get('role', 'user')).replace('human','user').replace('gpt','assistant'),
             'content': m.get('value', m.get('content', ''))} for m in conv]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, enable_thinking=True)
    except Exception:
        return '\n\n'.join(m['content'] for m in msgs)

def _text_openmath(row):
    q = row.get('problem') or row.get('question') or ''
    a = row.get('generated_solution') or row.get('solution') or row.get('answer') or ''
    return f'Problem: {q}\n\nSolution: {a}'

CORPUS_EXTRACTORS = {
    'HuggingFaceFW/fineweb-edu':       _text_fineweb,
    'open-thoughts/OpenThoughts-114k': _text_openthoughts,
    'nvidia/OpenMathInstruct-2':       _text_openmath,
}

def mixed_text_stream():
    streams = []
    weights = []
    for repo, split, w in CORPUS_MIX:
        it = iter(_load_stream(repo, split))
        streams.append((repo, it))
        weights.append(w)
    while True:
        idx = random.choices(range(len(streams)), weights=weights, k=1)[0]
        repo, it = streams[idx]
        try:
            row = next(it)
        except StopIteration:
            streams[idx] = (repo, iter(_load_stream(repo, CORPUS_MIX[idx][1])))
            row = next(streams[idx][1])
        txt = CORPUS_EXTRACTORS[repo](row)
        if txt and len(txt) > 50:
            yield txt

# Smoke test
_g = mixed_text_stream()
for _ in range(3):
    s = next(_g)
    print(f'[{len(s)} chars] {s[:100]!r}...')


## 7. LayerTap — one Qwen forward → 11 layer buffers


In [ ]:
class LayerTap:
    def __init__(self, model, layers):
        self.buf = {L: None for L in layers}
        self.handles = []
        for L in layers:
            mod = get_layer_module(model, L)
            self.handles.append(mod.register_forward_hook(self._mk_hook(L)))
    def _mk_hook(self, L):
        def _h(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.buf[L] = h.detach().to(torch.bfloat16)
        return _h
    def close(self):
        for h in self.handles:
            h.remove()

def pack_sequences(text_gen, n_seq, seq_len):
    out = []
    carry = []
    while len(out) < n_seq:
        if len(carry) < seq_len:
            carry.extend(tok(next(text_gen), add_special_tokens=False).input_ids)
            continue
        out.append(carry[:seq_len])
        carry = carry[seq_len:]
    return torch.tensor(out, dtype=torch.long, device=DEVICE)

def activation_stream(text_gen, token_budget):
    tap = LayerTap(model, LAYERS)
    emitted = 0
    try:
        while emitted < token_budget:
            ids = pack_sequences(text_gen, FWD_BATCH, SEQ_LEN)
            with torch.no_grad():
                model(ids)
            chunk = {L: tap.buf[L].reshape(-1, D_MODEL) for L in LAYERS}
            emitted += chunk[LAYERS[0]].shape[0]
            yield chunk, emitted
    finally:
        tap.close()

# Smoke test: one batch, all 11 layers
_g = mixed_text_stream()
_s = activation_stream(_g, token_budget=FWD_BATCH * SEQ_LEN)
_chunk, _em = next(_s)
for L in LAYERS:
    t = _chunk[L]
    print(f'L{L}: shape={tuple(t.shape)}, mean_norm={t.float().norm(dim=-1).mean():.3f}, std={t.float().std():.3f}')
del _g, _s, _chunk
torch.cuda.empty_cache()


## 8. TopK SAE with AuxK dead-feature loss (Gao et al. 2024)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TopKSAE(nn.Module):
    def __init__(self, d_in, n, k, k_aux):
        super().__init__()
        self.d_in, self.n, self.k, self.k_aux = d_in, n, k, k_aux
        W = torch.randn(n, d_in) / (d_in ** 0.5)
        W = W / W.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        self.W_dec = nn.Parameter(W)
        self.W_enc = nn.Parameter(W.T.clone().contiguous())
        self.b_enc = nn.Parameter(torch.zeros(n))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc

    def forward(self, x, dead_mask=None):
        pre = self.encode_pre(x)
        top_v, top_i = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, top_i, F.relu(top_v))
        x_hat = z @ self.W_dec + self.b_dec
        main_loss = (x - x_hat).pow(2).mean()
        aux_loss = x.new_zeros(())
        if dead_mask is not None and dead_mask.sum() >= self.k_aux:
            residual = (x - x_hat).detach()
            pre_dead = pre.masked_fill(~dead_mask.unsqueeze(0), float('-inf'))
            v_aux, i_aux = pre_dead.topk(self.k_aux, dim=-1)
            z_aux = torch.zeros_like(pre)
            z_aux.scatter_(-1, i_aux, F.relu(v_aux))
            res_hat = z_aux @ self.W_dec
            aux_loss = (residual - res_hat).pow(2).mean()
        return x_hat, z, top_i, main_loss, aux_loss

    @torch.no_grad()
    def renorm_decoder(self):
        self.W_dec.data /= self.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)

    @torch.no_grad()
    def set_b_dec_geomedian(self, samples, n_iter=40):
        med = samples.mean(0)
        for _ in range(n_iter):
            d = (samples - med).norm(dim=-1).clamp_min(1e-8)
            w = 1.0 / d
            med = (samples * w.unsqueeze(1)).sum(0) / w.sum()
        self.b_dec.data = med.to(self.b_dec.dtype)

# Smoke
_sae = TopKSAE(D_MODEL, 4096, 32, 256).to(DEVICE, DTYPE_SAE)
_x = torch.randn(64, D_MODEL, device=DEVICE, dtype=DTYPE_SAE)
_dm = torch.zeros(4096, dtype=torch.bool, device=DEVICE); _dm[:500] = True
_xh, _z, _ti, _ml, _al = _sae(_x, _dm)
print(f'smoke: x_hat={tuple(_xh.shape)}, main={_ml:.4f}, aux={_al:.4f}, L0={(_z>0).sum(-1).float().mean():.0f}')
del _sae, _x, _dm, _xh, _z, _ti, _ml, _al
torch.cuda.empty_cache()


## 9. Checkpoint utilities — Drive + HF (HARD RULE: dual save)

Every checkpoint goes to **two places**:
1. Drive (`OUT/sae_ckpt/sae_L{L}_resume.pt`) — primary, survives Colab disconnect
2. HF Hub (`HF_REPO/sae_L{L}_resume.pt`) — secondary, public artifact

On resume, load Drive first (faster), HF as fallback.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import save_file, load_file

hfapi = HfApi()
try:
    hfapi.create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
    print(f'Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'create_repo warning: {e}')

def _ckpt_paths(L):
    return dict(
        weights = f'sae_L{L}_latest.safetensors',
        resume  = f'sae_L{L}_resume.pt',
        cfg     = f'sae_L{L}_cfg.json',
    )

def save_ckpt(L, sae, optim, scheduler, step, tokens_seen, last_fired):
    """Save weights + resume state to BOTH Drive and HF."""
    p = _ckpt_paths(L)

    # 1. sae_lens-format weights (safetensors)
    weights_local = f'{LOCAL_CKPT_DIR}/{p["weights"]}'
    save_file({
        'W_enc': sae.W_enc.detach().cpu().contiguous(),
        'W_dec': sae.W_dec.detach().cpu().contiguous(),
        'b_enc': sae.b_enc.detach().cpu().contiguous(),
        'b_dec': sae.b_dec.detach().cpu().contiguous(),
    }, weights_local)

    # 2. cfg.json (sae_lens / Neuronpedia compatible)
    cfg_local = f'{LOCAL_CKPT_DIR}/{p["cfg"]}'
    with open(cfg_local, 'w') as f:
        json.dump({
            'architecture': 'topk',
            'd_in': D_MODEL,
            'd_sae': N_FEATURES,
            'k': K_TOPK,
            'k_aux': K_AUX,
            'alpha_aux': ALPHA_AUX,
            'hook_name': f'model.language_model.layers.{L}',
            'model_name': MODEL_ID,
            'activation_fn_str': 'relu',
            'tokens_seen': tokens_seen,
            'step': step,
        }, f, indent=2)

    # 3. Resume state (optimizer + scheduler + dead tracker)
    resume_local = f'{LOCAL_CKPT_DIR}/{p["resume"]}'
    torch.save({
        'sae_state': sae.state_dict(),
        'optim': optim.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'step': step,
        'tokens_seen': tokens_seen,
        'last_fired': last_fired.cpu(),
    }, resume_local)

    # 4a. Mirror to DRIVE (HARD RULE — primary persistence)
    for fname in [p['weights'], p['cfg'], p['resume']]:
        try:
            shutil.copy(f'{LOCAL_CKPT_DIR}/{fname}', OUT / 'sae_ckpt' / fname)
        except Exception as e:
            print(f'  Drive copy fail {fname}: {e}')

    # 4b. Mirror to HF (secondary, public artifact)
    for fname in [p['weights'], p['cfg'], p['resume']]:
        try:
            hfapi.upload_file(
                path_or_fileobj=f'{LOCAL_CKPT_DIR}/{fname}',
                path_in_repo=fname,
                repo_id=HF_REPO,
                commit_message=f'L{L} ckpt step={step} tokens={tokens_seen/1e6:.1f}M',
            )
        except Exception as e:
            print(f'  HF upload fail {fname}: {e}')

def load_ckpt(L, sae, optim, scheduler):
    """Try Drive → HF → fresh."""
    p = _ckpt_paths(L)

    # Try Drive first (fastest)
    drive_resume = OUT / 'sae_ckpt' / p['resume']
    if drive_resume.exists():
        try:
            state = torch.load(drive_resume, map_location='cpu', weights_only=False)
            sae.load_state_dict(state['sae_state'])
            optim.load_state_dict(state['optim'])
            if scheduler is not None and state.get('scheduler'):
                scheduler.load_state_dict(state['scheduler'])
            print(f'  ✓ resumed L{L} from Drive: step={state["step"]} tokens={state["tokens_seen"]/1e6:.1f}M')
            return state['step'], state['tokens_seen'], state['last_fired'].to(DEVICE)
        except Exception as e:
            print(f'  Drive resume failed L{L}: {e} — trying HF')

    # Fall back to HF
    try:
        local = hf_hub_download(HF_REPO, p['resume'], local_dir=LOCAL_CKPT_DIR)
        state = torch.load(local, map_location='cpu', weights_only=False)
        sae.load_state_dict(state['sae_state'])
        optim.load_state_dict(state['optim'])
        if scheduler is not None and state.get('scheduler'):
            scheduler.load_state_dict(state['scheduler'])
        print(f'  ✓ resumed L{L} from HF: step={state["step"]} tokens={state["tokens_seen"]/1e6:.1f}M')
        return state['step'], state['tokens_seen'], state['last_fired'].to(DEVICE)
    except Exception as e:
        print(f'  (no ckpt for L{L}, fresh init: {type(e).__name__})')
        return 0, 0, None

print('Checkpoint utilities ready (Drive primary, HF secondary).')


## 10. Init 11 SAEs + optimizers + schedulers + geomedian b_dec

All 11 SAEs share one Qwen forward pass via LayerTap.
Fresh SAEs get geomedian b_dec init (Weiszfeld) for heavy-tailed residual robustness.


In [ ]:
from torch.optim.lr_scheduler import LambdaLR

total_steps = TOKEN_BUDGET // BATCH_SIZE
print(f'Planned steps per SAE: {total_steps:,} ({TOKEN_BUDGET/1e6:.0f}M tokens)')

saes, optims, scheds, last_fired_map = {}, {}, {}, {}
for L in LAYERS:
    sae = TopKSAE(D_MODEL, N_FEATURES, K_TOPK, K_AUX).to(DEVICE, DTYPE_SAE)
    # torch.optim.Adam — state preserves bf16 param dtype (vs bnb's fp32 init transient)
    optim = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999), eps=1e-8)
    def lr_lambda(step, warm=WARMUP_STEPS, total=total_steps, floor=LR_FLOOR/LR_PEAK):
        if step < warm:
            return step / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    sched = LambdaLR(optim, lr_lambda)
    saes[L], optims[L], scheds[L] = sae, optim, sched

print(f'VRAM after SAE init: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Try resuming from Drive/HF
resume_steps, resume_tokens = {}, {}
for L in LAYERS:
    s, t, lf = load_ckpt(L, saes[L], optims[L], scheds[L])
    resume_steps[L] = s
    resume_tokens[L] = t
    last_fired_map[L] = lf if lf is not None else torch.full((N_FEATURES,), -DEAD_TOKENS, device=DEVICE, dtype=torch.long)

# Geomedian init for fresh layers
fresh_layers = [L for L in LAYERS if resume_steps[L] == 0]
if fresh_layers:
    print(f'\nGeomedian init for fresh layers: {fresh_layers}')
    _g = mixed_text_stream()
    _samples = {L: [] for L in fresh_layers}
    _collected = 0
    for chunk, emitted in activation_stream(_g, token_budget=16_384):
        for L in fresh_layers:
            _samples[L].append(chunk[L])
        _collected = emitted
        if _collected >= 16_384:
            break
    for L in fresh_layers:
        S = torch.cat(_samples[L])[:16_384].to(DEVICE, DTYPE_SAE)
        saes[L].set_b_dec_geomedian(S)
        print(f'  L{L} b_dec norm={saes[L].b_dec.norm():.3f}')
    del _samples
    torch.cuda.empty_cache()


## 11. Training loop — 11 SAEs, shared forward pass

**Resume-safe**: re-running this cell picks up from latest checkpoint.
**Drive checkpoint** every 10M tokens (HARD RULE).

Expected runtime: ~36-40h on RTX 6000 Blackwell 96GB.


In [ ]:
from tqdm.auto import tqdm
import time

start_tokens = min(resume_tokens.values()) if resume_tokens else 0
print(f'Resume from {start_tokens/1e6:.1f}M tokens → target {TOKEN_BUDGET/1e6:.0f}M per layer')

text_gen = mixed_text_stream()
remaining = TOKEN_BUDGET - start_tokens

global_tokens = start_tokens
last_ckpt_tokens = start_tokens
log_every_steps = 100
t0 = time.time()

metrics = {L: {'step': 0, 'main': 0.0, 'aux': 0.0, 'var_expl': 0.0, 'dead': 0} for L in LAYERS}

pbar = tqdm(total=remaining, unit='tok', unit_scale=True, smoothing=0.1, desc='SAE train (11 layers)')

for chunk, emitted_cum in activation_stream(text_gen, token_budget=remaining):
    batch_tokens = chunk[LAYERS[0]].shape[0]
    global_tokens += batch_tokens

    for L in LAYERS:
        sae, opt, sch = saes[L], optims[L], scheds[L]
        x = chunk[L].to(DEVICE, DTYPE_SAE, non_blocking=True)

        dead_mask = (last_fired_map[L] < (global_tokens - DEAD_TOKENS))

        perm = torch.randperm(x.shape[0], device=DEVICE)
        x = x[perm]
        for i in range(0, x.shape[0], BATCH_SIZE):
            xb = x[i:i+BATCH_SIZE]
            if xb.shape[0] < 64:
                continue
            xh, z, top_i, main_l, aux_l = sae(xb, dead_mask)
            loss = main_l + ALPHA_AUX * aux_l
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), GRAD_CLIP)
            opt.step()
            sch.step()
            sae.renorm_decoder()

            fired = torch.unique(top_i)
            last_fired_map[L][fired] = global_tokens

            with torch.no_grad():
                var = 1.0 - (xb - xh).pow(2).sum() / (xb - xb.mean(0)).pow(2).sum().clamp_min(1e-8)
            m = metrics[L]
            m['step'] += 1
            m['main'] = 0.98 * m['main'] + 0.02 * main_l.item() if m['step'] > 1 else main_l.item()
            m['aux']  = 0.98 * m['aux']  + 0.02 * (aux_l.item() if torch.is_tensor(aux_l) else aux_l) if m['step'] > 1 else (aux_l.item() if torch.is_tensor(aux_l) else 0.0)
            m['var_expl'] = 0.98 * m['var_expl'] + 0.02 * var.item() if m['step'] > 1 else var.item()
            m['dead'] = int(dead_mask.sum().item())

    pbar.update(batch_tokens)

    # Compact log: show 4 worst layers only
    if metrics[LAYERS[0]]['step'] % log_every_steps == 0:
        worst = sorted(LAYERS, key=lambda L: metrics[L]['var_expl'])[:4]
        msg = ' | '.join([f'L{L}:ve={metrics[L]["var_expl"]:.3f}' for L in worst])
        pbar.set_postfix_str(msg)

    # Checkpoint (Drive + HF every 10M tokens)
    if global_tokens - last_ckpt_tokens >= CKPT_EVERY_TOK:
        pbar.write(f'=== Checkpoint @ {global_tokens/1e6:.1f}M tokens ===')
        for L in LAYERS:
            save_ckpt(L, saes[L], optims[L], scheds[L],
                      step=metrics[L]['step'],
                      tokens_seen=global_tokens,
                      last_fired=last_fired_map[L])
        last_ckpt_tokens = global_tokens
        # Snapshot metrics to Drive
        with open(OUT / 'logs' / f'metrics_{global_tokens//1_000_000}M.json', 'w') as f:
            json.dump({str(L): metrics[L] for L in LAYERS}, f, indent=2)

    if global_tokens >= TOKEN_BUDGET:
        break

pbar.close()

# Final checkpoint
print('=== Final checkpoint ===')
for L in LAYERS:
    save_ckpt(L, saes[L], optims[L], scheds[L],
              step=metrics[L]['step'],
              tokens_seen=global_tokens,
              last_fired=last_fired_map[L])

elapsed = time.time() - t0
print(f'Total training time: {elapsed/3600:.2f}h ({global_tokens/1e6:.0f}M tokens × {len(LAYERS)} SAEs)')


## 12. Held-out validation (1M tokens) — var_expl + L0 + dead


In [ ]:
random.seed(SEED + 1)
val_gen = mixed_text_stream()

val_stats = {L: {'mse': 0.0, 'var_total': 0.0, 'residual_sq': 0.0, 'l0_sum': 0, 'l0_n': 0, 'fired': set()} for L in LAYERS}
VAL_TOKENS = 1_000_000

for chunk, emitted in activation_stream(val_gen, token_budget=VAL_TOKENS):
    for L in LAYERS:
        sae = saes[L]
        x = chunk[L].to(DEVICE, DTYPE_SAE)
        with torch.no_grad():
            xh, z, top_i, _, _ = sae(x)
            s = val_stats[L]
            s['residual_sq'] += (x - xh).pow(2).sum().item()
            s['var_total']   += (x - x.mean(0)).pow(2).sum().item()
            s['l0_sum']      += (z > 0).sum().item()
            s['l0_n']        += x.shape[0]
            s['fired'].update(top_i.unique().cpu().tolist())
    if emitted >= VAL_TOKENS:
        break

print(f'\nHeld-out validation ({VAL_TOKENS/1e6:.1f}M tokens):')
print(f'{"Layer":<6} {"var_expl":>10} {"L0":>8} {"alive":>8} {"dead%":>8}')
val_report = {}
for L in LAYERS:
    s = val_stats[L]
    var_expl = 1.0 - s['residual_sq'] / s['var_total']
    l0 = s['l0_sum'] / s['l0_n']
    alive = len(s['fired'])
    dead_pct = 100.0 * (1 - alive / N_FEATURES)
    val_report[L] = {'var_expl': var_expl, 'l0': l0, 'alive': alive, 'dead_pct': dead_pct}
    print(f'L{L:<5} {var_expl:>10.4f} {l0:>8.1f} {alive:>8d} {dead_pct:>7.2f}%')

# Save report to Drive + HF
report_path = OUT / 'val_report.json'
with open(report_path, 'w') as f:
    json.dump({str(L): v for L, v in val_report.items()}, f, indent=2)
shutil.copy(report_path, f'{LOCAL_CKPT_DIR}/val_report.json')
hfapi.upload_file(
    path_or_fileobj=f'{LOCAL_CKPT_DIR}/val_report.json',
    path_in_repo='val_report.json',
    repo_id=HF_REPO,
    commit_message=f'Held-out validation ({VAL_TOKENS/1e6:.0f}M tokens)',
)


## 13. Cleanup — remove resume.pt bloat, write README, finalize repo


In [ ]:
from huggingface_hub import list_repo_files

# Remove resume.pt (large, only needed during training)
try:
    files = list_repo_files(HF_REPO)
    for f in files:
        if f.endswith('_resume.pt'):
            try:
                hfapi.delete_file(f, repo_id=HF_REPO, commit_message='cleanup resume state after convergence')
                print(f'  deleted {f}')
            except Exception as e:
                print(f'  delete fail {f}: {e}')
except Exception as e:
    print(f'list warning: {e}')

# Write a README
readme_text = f'''---
license: apache-2.0
tags:
  - sparse-autoencoder
  - mechanistic-interpretability
  - qwen3.6
  - reasoning-models
  - pgac
---

# Qwen3.6-27B Full-Stack TopK SAEs

Companion to [`caiovicentino1/qwen36-27b-sae-papergrade`](https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade)
(L11/L31/L55). Together they cover **14 layers** at every-4th from L11→L63, enabling
Probe-Gated Adaptive Compute (PGAC) routing experiments.

## Architecture

- TopK SAE (Gao et al. 2024) with AuxK dead-feature loss
- d_in = 5120, d_sae = 65536 (13× expansion), k = 128
- Trained on 200M tokens per layer (fineweb-edu 70% / OpenThoughts 20% / OpenMathInstruct 10%)

## Layers

{json.dumps(LAYERS, indent=0)}

## Loading

```python
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

L = 23
weights = load_file(hf_hub_download('{HF_REPO}', f'sae_L{{L}}_latest.safetensors'))
# weights: dict with W_enc, W_dec, b_enc, b_dec
```

## Validation

See `val_report.json` for per-layer var_expl, L0, alive feature count.
'''

readme_path = f'{LOCAL_CKPT_DIR}/README.md'
with open(readme_path, 'w') as f:
    f.write(readme_text)
hfapi.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo='README.md',
    repo_id=HF_REPO,
    commit_message='Final README with companion-repo cross-link',
)

print('\nFinal repo layout:')
for f in sorted(list_repo_files(HF_REPO)):
    print(f'  {f}')

print(f'\n✓ Full-stack SAEs ready: https://huggingface.co/{HF_REPO}')
print(f'✓ Combined coverage: L11/L15/L19/L23/L27/L31/L35/L39/L43/L47/L51/L55/L59/L63 (14 layers)')
print(f'✓ Ready for Phase 3 — PGAC kernel benchmark')
